In [ ]:
import numpy as np

In [4]:
import time
import random
import numpy as np
import galois
from pylfsr import LFSR

# ============================================================
# Shared helpers
# ============================================================

def irreducible_polynomial(bH):
    GF = galois.GF(2)
    while True:
        coeffs = [1] + [random.randint(0, 1) for _ in range(bH)]
        p = galois.Poly(coeffs, field=GF)
        if p.is_irreducible():
            return coeffs


def coeffs_to_exps_original(coeffs, bH):
    exps = [bH]
    for i, c in enumerate(coeffs[1:-1], start=1):
        if c == 1:
            exps.append(bH - i)
    return exps

def coeffs_to_exps(coeffs, bH):
    exps = [] #[bH]
    for i, c in enumerate(coeffs[:-1]):
        if c == 1:
            exps.append(bH - i)
    return exps

# ============================================================
# Version 0: naive reference (ground truth for correctness checks)
# ============================================================

def toeplitz_naive(coeffs, state, bH, bM):
    exps = coeffs_to_exps(coeffs, bH)
    lfsr = LFSR(exps, state)
    columns = []
    for _ in range(bM):
        columns.append(list(lfsr.state))
        lfsr.next()
    T = np.column_stack(columns)
    return T


def matvec_naive(T, message):
    message = np.asarray(message, dtype=np.uint8)
    return (T @ message) % 2


# ============================================================
# Version 1: build T (preallocated), then bit-packed popcount matvec
# ============================================================

_POPCOUNT_LUT = np.array([bin(i).count("1") & 1 for i in range(256)], dtype=np.uint8)


def gf2_matvec_popcount(T, vec):
    T = np.ascontiguousarray(T, dtype=np.uint8)
    vec = np.ascontiguousarray(vec, dtype=np.uint8)

    Tp = np.packbits(T, axis=1)
    vp = np.packbits(vec)

    anded = Tp & vp
    parity_per_byte = _POPCOUNT_LUT[anded]
    return np.bitwise_xor.reduce(parity_per_byte, axis=1)


def toeplitz_prealloc(coeffs, state, bH, bM):
    exps = coeffs_to_exps(coeffs, bH)
    lfsr = LFSR(exps, state)
    T = np.empty((bH, bM), dtype=np.uint8)
    for j in range(bM):
        T[:, j] = lfsr.state
        #print(T[:, j])
        lfsr.next()
    return T


def sign_v1_popcount(coeffs, state, bH, message):
    T = toeplitz_prealloc(coeffs, state, bH, len(message))
    return gf2_matvec_popcount(T, message)


# ============================================================
# Version 2: fused, no T ever materialized
# ============================================================

def sign_v2_fused(coeffs, state, bH, message):
    exps = coeffs_to_exps(coeffs, bH)
    lfsr = LFSR(exps, state)

    message = np.asarray(message, dtype=np.uint8)
    acc = np.zeros(bH, dtype=np.uint8)

    for j in range(len(message)):
        if message[j]:
            acc ^= np.asarray(lfsr.state, dtype=np.uint8)
        lfsr.next()

    return acc

# ============================================================
# Version 3: remove pylfsr
# ============================================================

def toeplitz_raw_numpy(coeffs, state, bH, bM):
    """Same output as toeplitz_prealloc, but steps the LFSR manually
    in numpy instead of calling pylfsr.next() each iteration."""
    # Build the feedback-tap mask directly from coeffs, matching
    # coeffs_to_exps' convention (bH is always a tap; middle coeffs
    # optionally are).
    exps = coeffs_to_exps(coeffs, bH)  # e.g. [10, 8, 3]

    state = np.array(state, dtype=np.uint8)
    T = np.empty((bH, bM), dtype=np.uint8)

    # tap_indices: which positions of `state` (0-indexed from the
    # "s_{n-1}" end) contribute to the feedback bit. Verify against
    # pylfsr's actual state ordering before trusting this mapping.
    tap_indices = [e - 1 for e in exps]
    #print(tap_indices)
    #print(exps)

    for j in range(bM):
        T[:, j] = state
        #print(T[:, j], "v3")
        feedback = np.bitwise_xor.reduce(state[tap_indices])
        state = np.concatenate(([feedback], state[:-1]))

    return T


# ============================================================
# Correctness check
# ============================================================

def check_correctness(trials=20, bH=10, bM=500):
    print(f"Running {trials} correctness trials (bH={bH}, bM={bM})...")
    for t in range(trials):
        coeffs = irreducible_polynomial(bH)
        state = [random.randint(0, 1) for _ in range(bH)]
        message = [random.randint(0, 1) for _ in range(bM)]

        T_ref = toeplitz_naive(coeffs, state, bH, bM)
        expected = matvec_naive(T_ref, message)

        got_v1 = sign_v1_popcount(coeffs, state, bH, message)
        got_v2 = sign_v2_fused(coeffs, state, bH, message)
        got_v3 = matvec_naive(toeplitz_raw_numpy(coeffs, state, bH, bM), message)

        if not np.array_equal(expected, got_v1):
            print(f"  [FAIL] trial {t}: v1 (popcount) mismatch")
            print(f"    expected={expected}")
            print(f"    got     ={got_v1}")
            return False
        if not np.array_equal(expected, got_v2):
            print(f"  [FAIL] trial {t}: v2 (fused) mismatch")
            print(f"    expected={expected}")
            print(f"    got     ={got_v2}")
            return False

        if not np.array_equal(expected, got_v3):
            print(f"  [FAIL] trial {t}: v3 (numpy) mismatch")
            print(f"    expected={expected}")
            print(f"    got     ={got_v3}")
            return False

    print("  All trials passed. Both optimized versions match the naive reference.\n")
    return True


# ============================================================
# Benchmark
# ============================================================

def benchmark(bH_values, bM_values, repeats=5, newest=False):
    if newest is False:
        print(f"{'bH':>6} {'bM':>8} {'naive(s)':>12} {'popcount(s)':>14} {'fused(s)':>12} {'numpy(s)':>12}")
    else:
        
        print(f"{'bH':>6} {'bM':>8} {'numpy(s)':>12}")
    for bH in bH_values:
        for bM in bM_values:
            coeffs = irreducible_polynomial(bH)
            state = [random.randint(0, 1) for _ in range(bH)]
            message = [random.randint(0, 1) for _ in range(bM)]
            if newest is False:
                # naive
                t0 = time.perf_counter()
                for _ in range(repeats):
                    T = toeplitz_naive(coeffs, state, bH, bM)
                    matvec_naive(T, message)
                t_naive = (time.perf_counter() - t0) / repeats

                # popcount (build + bit-packed matvec)
                t0 = time.perf_counter()
                for _ in range(repeats):
                    sign_v1_popcount(coeffs, state, bH, message)
                t_popcount = (time.perf_counter() - t0) / repeats

                # fused
                t0 = time.perf_counter()
                for _ in range(repeats):
                    sign_v2_fused(coeffs, state, bH, message)
                t_fused = (time.perf_counter() - t0) / repeats

            #removed pylfsr
            t0 = time.perf_counter()
            for _ in range(repeats):
                T = toeplitz_raw_numpy(coeffs, state, bH, bM)
                matvec_naive(T, message)
            t_numpy = (time.perf_counter() - t0) / repeats

            if newest is False:
                print(f"{bH:>6} {bM:>8} {t_naive:>12.6f} {t_popcount:>14.6f} {t_fused:>12.6f} {t_numpy:>12.6f}")
            else:
                print(f"{bH:>6} {bM:>8} {t_numpy:>12.6f}")
            
if __name__ == "__main__":
    if not check_correctness(trials=20, bH=10, bM=500):
    #if not check_correctness(trials=1, bH=10, bM=10):
        raise SystemExit("Correctness check failed — do not trust benchmark results until fixed.")

    benchmark(
        #bH_values=[10, 32, 64],
        bM_values=[10000, 100000],
        #bH_values=[int(np.log2(x)) for x in [30, 50]],
        bH_values=[30, 50],
        repeats=5,
    )

Running 20 correctness trials (bH=10, bM=500)...
  All trials passed. Both optimized versions match the naive reference.

    bH       bM     naive(s)    popcount(s)     fused(s)     numpy(s)
    30    10000     0.251205       0.192813     0.197446     0.023965
    30   100000     2.769337       2.434310     2.492195     0.231941
    50    10000     0.334721       0.294242     0.287802     0.026988
    50   100000     4.005227       3.381380     3.410606     0.258996


In [2]:
if __name__ == "__main__":
    if not check_correctness(trials=20, bH=10, bM=500):
        raise SystemExit("Correctness check failed — do not trust benchmark results until fixed.")

    benchmark(
        #bH_values=[10, 32, 64],
        bM_values = [
            50_000,
            75_000,
            100_000,
            125_000,
            150_000,
            175_000,
            200_000,
            225_000,
            250_000,
            300_000,
        ],
        #bH_values=[int(np.log2(x)) for x in [30, 50]],
        bH_values=[30],
        repeats=3,
    )

Running 20 correctness trials (bH=10, bM=500)...
  All trials passed. Both optimized versions match the naive reference.

    bH       bM     naive(s)    popcount(s)     fused(s)     numpy(s)
    30    50000     1.520805       1.248434     1.304354     0.137333
    30    75000     2.393146       2.113328     2.132348     0.209050
    30   100000     3.533694       3.173200     3.282524     0.754189
    30   125000     5.982310       5.030936     4.970505     0.390728
    30   150000     7.390670       6.381055     5.928320     0.432714
    30   175000     8.500446       7.844258     7.847094     0.511848
    30   200000    10.142453       9.262565     9.544572     0.680224
    30   225000    14.610999      12.472233    12.942113     0.672155
    30   250000    18.155703      17.335788    16.589317     0.776801
    30   300000    27.973365      25.934341    26.377524     0.929918


In [5]:
if __name__ == "__main__":
    if not check_correctness(trials=20, bH=10, bM=500):
        raise SystemExit("Correctness check failed — do not trust benchmark results until fixed.")

    benchmark(
        #bH_values=[10, 32, 64],
        bM_values = [
            500_000,
            1_000_000,
            10_000_000,
            100_000_000,
            1_000_000_000
        ],
        #bH_values=[int(np.log2(x)) for x in [30, 50]],
        bH_values=[30],
        repeats=3,
        newest=True
    )

Running 20 correctness trials (bH=10, bM=500)...
  All trials passed. Both optimized versions match the naive reference.

    bH       bM     numpy(s)
    30   500000     1.216613
    30  1000000     2.475347
    30 10000000    24.614091
    30 100000000   287.851445


KeyboardInterrupt: 

In [ ]:
def irreducible_polynomial(bH):
    GF = galois.GF(2)
    while True:
        coeffs = [1] + [random.randint(0, 1) for _ in range(bH)]
        p = galois.Poly(coeffs, field=GF)
        if p.is_irreducible():
            return coeffs


def coeffs_to_exps(coeffs, bH):
    exps = [bH]
    for i, c in enumerate(coeffs[1:-1], start=1):
        if c == 1:
            exps.append(bH - i)
    return exps

def coeffs_to_exps_v2(coeffs, bH):
    exps = [] #[bH]
    for i, c in enumerate(coeffs[:-1]):
        if c == 1:
            exps.append(bH - i)
    return exps

def coeffs_to_exps_v3(coeffs, bH):
    exps = [] #[bH]
    for i, c in enumerate(coeffs[:-1]):
        if c == 1:
            exps.append(bH - i)
    return exps

bH = 10
state = [0,1,0,1,0,1,0,1,0,1]
coeffs = irreducible_polynomial(10)
exps = coeffs_to_exps(coeffs, bH)
exps2 = coeffs_to_exps_v2(coeffs, bH)
print(coeffs, exps, exps2)
lfsr = LFSR(exps, state)
print(lfsr.state)
lfsr.next()
print(lfsr.state)

[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1] [10, 9, 8, 7, 6, 5, 4, 3] [10, 9, 8, 7, 6, 5, 4, 3]
[0 1 0 1 0 1 0 1 0 1]
[0 1 0 1 0 1 0 1 0 1]


In [53]:
s = [1,0,1,1]
fpoly = [3, 4]
lfsr = LFSR(fpoly, s)
print(lfsr.state)
lfsr.next()
print(lfsr.state)
lfsr.next()
print(lfsr.state)
lfsr.next()
print(lfsr.state)
lfsr.next()
print(lfsr.state)
lfsr.next()
print(lfsr.state)
lfsr.next()
print(lfsr.state)

[1 0 1 1]
[0 1 0 1]
[1 0 1 0]
[1 1 0 1]
[1 1 1 0]
[1 1 1 1]
[0 1 1 1]


In [ ]:
def toeplitz_raw_numpy(coeffs, state, bH, bM):
    """Same output as toeplitz_prealloc, but steps the LFSR manually
    in numpy instead of calling pylfsr.next() each iteration."""
    # Build the feedback-tap mask directly from coeffs, matching
    # coeffs_to_exps' convention (bH is always a tap; middle coeffs
    # optionally are).
    exps = coeffs_to_exps(coeffs, bH)  # e.g. [10, 8, 3]

    state = np.array(state, dtype=np.uint8)
    T = np.empty((bH, bM), dtype=np.uint8)

    # tap_indices: which positions of `state` (0-indexed from the
    # "s_{n-1}" end) contribute to the feedback bit. Verify against
    # pylfsr's actual state ordering before trusting this mapping.
    tap_indices = [bH - e for e in exps]

    for j in range(bM):
        T[:, j] = state
        feedback = np.bitwise_xor.reduce(state[tap_indices])
        state = np.concatenate((state[1:], [feedback]))

    return T